In [ ]:
from pathlib import Path
import mimetypes

exclude = {"DevTools", ".github", ".godot"}
PROJECT_DIR = Path("..").resolve()


def is_code(path: Path) -> bool:
  if path.suffix in (".tscn", ".gd", ".tres", ".gdshader", ".uid"):
    return True
  return False

def is_text_file(path: Path) -> bool:
  mime, _ = mimetypes.guess_type(path)
  return mime is not None and mime.startswith("text/")

def ends_with_newline(path: Path) -> bool:
  """Check whether the file ends with a newline."""
  try:
    with path.open("rb") as f:
      if f.seek(0, 2) == 0:  # empty file
        return True
      f.seek(-1, 2)
      last_byte = f.read(1)
    return last_byte == b"\n"
  except Exception:
    return True  # play safe


def add_newline(path: Path):
  """Append newline to file."""
  with path.open("a", encoding="utf-8", newline="") as f:
    f.write("\n")

def normalize_newlines(path: Path) -> bool:
  """Convert CRLF to LF. Returns True if file changed."""
  try:
    text = path.read_text(encoding="utf-8", newline="")
    new_text = text.replace("\r\n", "\n")
    if new_text != text:
      path.write_text(new_text, encoding="utf-8", newline="") # note: newline="\n" and newline="" only preserves current line endings (does not convert)
      return True
    return False
  except Exception:
    return False


def strip_trailing_spaces(path: Path) -> bool:
  """Remove trailing spaces from each line. Returns True if file changed."""
  changed = False
  try:
    with path.open("r", encoding="utf-8", newline="") as f:
      lines = f.readlines()

    new_lines = []
    for line in lines:
      stripped = line.rstrip("\n \t")
      if stripped != line.rstrip("\n"):
        changed = True
      new_lines.append(stripped + ("\n" if line.endswith("\n") else ""))

    if changed:
      with path.open("w", encoding="utf-8", newline="") as f:
        f.writelines(new_lines)

    return changed
  except Exception:
    return False

for path in PROJECT_DIR.rglob("*"):
# for path in PROJECT_DIR.glob("**/*.uid"):
  if path.relative_to(PROJECT_DIR).parts[0] in set(["DevTools", ".github", ".godot"]):
    continue

  if not path.is_file() or not (is_code(path) or is_text_file(path)):
    continue

  # Remove trailing spaces (only for code files)
  if is_code(path):
    if normalize_newlines(path):
      print(f"Normalized line endings: {path}")
    if strip_trailing_spaces(path):
      print(f"Trimmed trailing spaces: {path}")

  if not ends_with_newline(path):
    print(f"Added newline at the end: {path}")
    add_newline(path)